# 取り込みのパターン: 5つの入口から1つのテーブルへ

データが1か所から届くデスクはありません。ティックフィードは Arrow のバッチを寄こし、
リサーチのノートブックは pandas や polars に住み、ベンダーは Parquet を置いていき、
どこかの古いプロセスはいまだに CSV をメールで送ってきます。

h5i-db の取り込み面は意図的に小さくしてあります。`append` がフィードを伸ばし、`write` が
中身を置き換える。どちらも Arrow の形をしたものなら何でも受け取ります。

このレシピで進めるのは次の4つです。

1. 連続する5セッションを5つの別々のソースから1つの `trades` テーブルに取り込む
2. `write` と `append` の意味論を対比する
3. 楽観ロックで、同時に走るローダを安全にする
4. コミットをまとめ、残ったセグメントを圧縮する

In [1]:
import shutil
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

import h5i_db
from h5i_db import col, count_star, time_bucket
import cookbook_utils as cu

db = h5i_db.Database(cu.fresh_db("00_ingestion"), create=True)

## データ

`cu.make_trades` が返す、連続11セッションぶんのティックデータです。1行が1約定です。

| 列 | 型 | 意味 |
| --- | --- | --- |
| `ts` | `timestamp[us, tz=UTC]` | 約定時刻、昇順 |
| `symbol` | `string` | 銘柄コード |
| `price` | `float64` | 約定価格 |
| `size` | `int64` | 約定株数 |
| `exchange` | `string` | 報告した取引所 |
| `side` | `string` | `B` は買い主導、`S` は売り主導 |

In [2]:
tape = cu.make_trades(days=11, trades_per_day=4_000, start="2026-06-01", seed=7)
print(f"{tape.num_rows:,} rows x {tape.num_columns} columns")
tape.to_pandas().head()

141,301 rows x 6 columns


,ts,symbol,price,size,exchange,side
0,2026-06-01 13:30:00.032537+00:00,NVDA,319.19,1,ARCA,S
1,2026-06-01 13:30:00.436084+00:00,NVDA,319.19,1,BATS,S
2,2026-06-01 13:30:00.729962+00:00,AAPL,265.08,1,IEX,B
3,2026-06-01 13:30:01.122309+00:00,MSFT,363.04,1,BATS,B
4,2026-06-01 13:30:01.757669+00:00,MSFT,362.94,100,ARCA,S


これを日ごとのバッチに切り分け、「1回の配信」が時刻順に届くようにします。`append` は
どのバッチも、テーブルに保存済みの最大タイムスタンプ以降から始まることを要求します。
フィードの意味論とは、実務上はそういうことです。ベンダーの受け渡し場所の代わりに、
`data/dbs` の下にステージング用のディレクトリを1つ用意します。

In [3]:
dates = tape["ts"].to_pandas().dt.date
sessions = sorted(dates.unique())
by_day = {d: tape.filter(pa.array((dates == d).to_numpy())) for d in sessions}
print(f"{len(sessions)} sessions, {len(tape):,} trades:", sessions[0], "→", sessions[-1])

staging = Path("data/dbs/00_ingestion_staging")
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)

11 sessions, 141,301 trades: 2026-06-01 → 2026-06-15


行き先は1つのテーブルです。以下のソースはすべてここに着地します。

In [4]:
SCHEMA = pa.schema(
    [
        pa.field("ts", pa.timestamp("us", tz="UTC"), nullable=False),
        pa.field("symbol", pa.string()),
        pa.field("price", pa.float64()),
        pa.field("size", pa.int64()),
        pa.field("exchange", pa.string()),
        pa.field("side", pa.string()),
    ]
)
db.create_table("trades", SCHEMA, time_column="ts", sort_key=["ts", "symbol"])

{'table': 'trades',
 'sequence': 0,
 'op': 'create',
 'rows_total': 0,
 'segments_total': 0,
 'segments_added': 0,
 'segments_deduped': 0,
 'committed_at_ns': 1785195806496479372}

## 1. pyarrow の Table から

変換の要らないネイティブの経路です。`append` はどれも**コミット辞書**を返します。新しい
バージョン番号（`sequence`）と、コミット後の行数・セグメント数が入っています。ローダでは
これをログに残してください。配信とバージョンを結びつける受領証です。

In [5]:
commit = db.append("trades", by_day[sessions[0]], note="day 1: arrow feed")
commit

{'table': 'trades',
 'sequence': 1,
 'op': 'append',
 'rows_total': 11418,
 'segments_total': 1,
 'segments_added': 1,
 'segments_deduped': 0,
 'committed_at_ns': 1785195806519109667}

## 2. pandas の DataFrame から

重い仕事は `pa.Table.from_pandas` がやってくれます。それに加えて `schema=` を必ず渡して
ください。pandas のバージョンによっては、日時がテーブルのマイクロ秒ではなくナノ秒として
往復してしまい、厳格な append がその不一致を拒否します。目的のスキーマを渡しておけば、
変換のついでにキャストされます。

In [6]:
df = by_day[sessions[1]].to_pandas()  # pretend this came from research code
commit = db.append(
    "trades",
    pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False),
    note="day 2: pandas",
)
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 2, 'rows_total': 25773, 'segments_total': 2}

## 3. polars の DataFrame から

polars は Arrow をそのまま話しますが、1つだけ癖があります。`to_arrow()` が吐くのは
`large_string` の列で、厳格な append はこれをスキーマ不一致として弾きます。`.cast(SCHEMA)`
ならメタデータ層で片付くので、コストはかかりません。polars から h5i-db へ渡す境界では、
これを習慣にしてください。

In [7]:
pldf = pl.from_arrow(by_day[sessions[2]])  # pretend this came from a polars pipeline
commit = db.append("trades", pldf.to_arrow().cast(SCHEMA), note="day 3: polars")
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 3, 'rows_total': 38842, 'segments_total': 3}

## 4. Parquet ファイルから

Parquet は型をそのまま保つので、ベンダーが置いていった Parquet は読んで append するだけ
です。行の並び順が怪しいベンダーなら、append の前にソートしておきましょう。どちらにせよ
厳格な append が教えてくれます。

In [8]:
pq.write_table(by_day[sessions[3]], staging / "vendor_day4.parquet")

commit = db.append("trades", pq.read_table(staging / "vendor_day4.parquet"), note="day 4: parquet drop")
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 4, 'rows_total': 49876, 'segments_total': 4}

## 5. CSV から

CSV は値を保ちますが型を落とします。素直に読むと `ts` はパーサが推測した何かとして返って
きます。`ConvertOptions(column_types=...)` でパース時に時刻列を `timestamp[us, tz=UTC]` に
固定すれば、スキーマは厳密に一致します。

In [9]:
pacsv.write_csv(by_day[sessions[4]], staging / "legacy_day5.csv")

from_csv = pacsv.read_csv(
    staging / "legacy_day5.csv",
    convert_options=pacsv.ConvertOptions(column_types={"ts": pa.timestamp("us", tz="UTC")}),
)
assert from_csv.schema.equals(by_day[sessions[4]].schema)
commit = db.append("trades", from_csv, note="day 5: legacy csv")
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 5, 'rows_total': 63593, 'segments_total': 5}

## `write` と `append`

- **`append`** はフィードを伸ばします。厳密に時刻順で、既存の行には触れません。テープの
  ように振る舞うものはすべてこちらです。
- **`write`** はテーブルの中身を、渡したデータで置き換えます。ただし*新しいバージョン*と
  してで、履歴はすべて残ります。まるごと言い直される参照データ、たとえばユニバースの構成、
  銘柄マッピング、リスクリミットはこちらです。

どちらも破壊的ではありません。古いバージョンは Python の `db.read(..., version=n)` からも、
SQL の `h5i('table', n)` からも読めるままです。

In [10]:
universe_schema = pa.schema(
    [pa.field("ts", pa.timestamp("us", tz="UTC"), nullable=False), pa.field("symbol", pa.string())]
)
db.create_table("universe", universe_schema, time_column="ts")

asof = pa.scalar(pd.Timestamp("2026-06-01", tz="UTC"), type=pa.timestamp("us", tz="UTC"))
db.write(
    "universe",
    pa.table({"ts": pa.array([asof] * 3), "symbol": pa.array(["AAPL", "MSFT", "NVDA"])}),
    note="June universe",
)
db.write(
    "universe",
    pa.table({"ts": pa.array([asof] * 4), "symbol": pa.array(["AAPL", "MSFT", "NVDA", "AVGO"])}),
    note="June universe, AVGO added",
)
print("head :", db.read("universe")["symbol"].to_pylist())
print("v1   :", db.read("universe", version=1)["symbol"].to_pylist())
[{k: v[k] for k in ("sequence", "op", "rows", "note") if k in v} for v in db.versions("universe")]

head : ['AAPL', 'MSFT', 'NVDA', 'AVGO']
v1   : ['AAPL', 'MSFT', 'NVDA']


[{'sequence': 0, 'op': 'create', 'rows': 0},
 {'sequence': 1, 'op': 'write', 'rows': 3, 'note': 'June universe'},
 {'sequence': 2,
  'op': 'write',
  'rows': 4,
  'note': 'June universe, AVGO added'}]

## `expected_version` による楽観ロック

1つのテーブルを2つのローダが共有していると、「いつでも好きなものを append する」は配信を
静かに混ぜ込みます。`append(..., expected_version=n)` はコミットを compare-and-swap に
変えます。テーブルの先頭がまだバージョン `n` のときだけ着地するのです。そうでなければ
`ConflictError` が返り、`retryable=True` と復旧手順を書いたヒントが付いてきます。リトライ
自体は機械的です。先頭を読み直して、もう一度 append するだけです。

In [11]:
day6 = by_day[sessions[5]]
try:
    db.append("trades", day6, expected_version=1)  # stale: head is already at v5
except h5i_db.ConflictError as e:
    print(f"code      {e.code}")
    print(f"retryable {e.retryable}")
    print(f"hint      {e.hint}")

# retry pattern: re-read the head version, then re-append against it
head = db.versions("trades")[-1]["sequence"]
commit = db.append("trades", day6, expected_version=head, note="day 6: CAS append")
print(f"\nretried against v{head} -> committed v{commit['sequence']}")

code      version_conflict
retryable True
hint      re-read the head of "trades" and retry against it; pure appends rebase safely (the CLI and Python bindings already auto-retry those)

retried against v5 -> committed v6


## バッチ化と圧縮

コミット1件ごとにマニフェストと最低1つのセグメントが書かれます。だから*バッチ*でコミット
してください。1日ぶん、1時間ぶん、数千行ぶん。1行ずつは絶対にやめましょう。

1日単位のコミットでも小さなセグメントは溜まりますし、クエリの計画はその全部に触ります。
下のループが普通のリズムです。平日のあいだは小さな append コミットを重ね、最後に `compact`
を1回かけてセグメントをまとめます。圧縮もそれ自体がただのコミットで、同じデータを少ない
セグメントで持ち、履歴はすべて残ります。

In [12]:
for d in sessions[6:]:
    commit = db.append("trades", by_day[d], note=f"daily load {d}")
    print(f"v{commit['sequence']}: +{len(by_day[d]):>6,} rows "
          f"-> {commit['segments_total']:>2} segments total")

v7: +13,806 rows ->  7 segments total
v8: +12,169 rows ->  8 segments total
v9: +11,128 rows ->  9 segments total
v10: +13,093 rows -> 10 segments total
v11: +14,464 rows -> 11 segments total


In [13]:
before = db.versions("trades")[-1]
commit = db.compact("trades")
print(f"compacted: {before['segments']} segments -> {commit['segments_total']}, "
      f"rows unchanged: {commit['rows_total']:,}")

[
    {k: v[k] for k in ("sequence", "op", "rows", "segments", "note") if k in v}
    for v in db.versions("trades")
]

compacted: 11 segments -> 1, rows unchanged: 141,301


[{'sequence': 0, 'op': 'create', 'rows': 0, 'segments': 0},
 {'sequence': 1,
  'op': 'append',
  'rows': 11418,
  'segments': 1,
  'note': 'day 1: arrow feed'},
 {'sequence': 2,
  'op': 'append',
  'rows': 25773,
  'segments': 2,
  'note': 'day 2: pandas'},
 {'sequence': 3,
  'op': 'append',
  'rows': 38842,
  'segments': 3,
  'note': 'day 3: polars'},
 {'sequence': 4,
  'op': 'append',
  'rows': 49876,
  'segments': 4,
  'note': 'day 4: parquet drop'},
 {'sequence': 5,
  'op': 'append',
  'rows': 63593,
  'segments': 5,
  'note': 'day 5: legacy csv'},
 {'sequence': 6,
  'op': 'append',
  'rows': 76641,
  'segments': 6,
  'note': 'day 6: CAS append'},
 {'sequence': 7,
  'op': 'append',
  'rows': 90447,
  'segments': 7,
  'note': 'daily load 2026-06-09'},
 {'sequence': 8,
  'op': 'append',
  'rows': 102616,
  'segments': 8,
  'note': 'daily load 2026-06-10'},
 {'sequence': 9,
  'op': 'append',
  'rows': 113744,
  'segments': 9,
  'note': 'daily load 2026-06-11'},
 {'sequence': 10,
  'op

In [14]:
# One tape, five formats, eleven commits - and a query sees a single clean table.
(
    db.table("trades")
    .group_by(time_bucket("1d", col("ts")).alias("session"))
    .agg(
        trades=count_star(),
        notional_mm=((col("price") * col("size")).sum() / 1e6).round(1),
    )
    .sort("session")
    .to_pandas()
)

,session,trades,notional_mm
0,2026-06-01 00:00:00+00:00,11418,250.4
1,2026-06-02 00:00:00+00:00,14355,339.6
2,2026-06-03 00:00:00+00:00,13069,295.4
3,2026-06-04 00:00:00+00:00,11034,252.4
4,2026-06-05 00:00:00+00:00,13717,309.5
5,2026-06-08 00:00:00+00:00,13048,294.2
6,2026-06-09 00:00:00+00:00,13806,319.5
7,2026-06-10 00:00:00+00:00,12169,264.3
8,2026-06-11 00:00:00+00:00,11128,251.3
9,2026-06-12 00:00:00+00:00,13093,288.0


## まとめ

- Arrow の形をしたものはそのまま append できます。境界で引っかかるのは pandas のナノ秒
  （`from_pandas(schema=...)`）、polars の `large_string`（`.cast(schema)`）、CSV の型忘れ
  （`ConvertOptions`）の3つです。
- `append` は厳密な時刻順でフィードを伸ばし、`write` は中身をまるごと新しいバージョンとして
  言い直します。どちらも履歴を壊しません。
- コミット辞書（`sequence`、`rows_total`、`segments_total`）が取り込みの受領証です。ログに
  残してください。
- `expected_version` は append を compare-and-swap に変えます。リトライ可能な
  `ConflictError` が出たら、先頭を読み直して append し直します。
- コミットはまとめ、小さな append が続いたあとは `compact` をかけます。圧縮は履歴に触れず
  セグメントを併合するだけの、ごく普通のバージョンです。

In [15]:
db.close()